# 13 Clustering and Anomaly Detection

This copy notebook keeps the unsupervised stage aligned with the course workflow.

Allowed methods used here:
- KMeans
- Agglomerative clustering
- DBSCAN
- Gaussian Mixture Models
- Isolation Forest
- Local Outlier Factor

The unit of analysis is the flood event. Clusters are exploratory archetypes, and anomaly flags are diagnostics rather than data-cleaning rules.

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [3]:
from project_name.modeling_stage import (
    ANOMALY_SCORES_PATH,
    CLUSTERING_ARCHETYPES_GEOPARQUET_PATH,
    CLUSTERING_ARCHETYPES_PATH,
    CLUSTERING_MODEL_SELECTION_PATH,
    clustering_feature_sets,
    evaluate_clustering_algorithms,
    feature_catalog,
    load_modeling_frame,
    pca_embedding,
    summarize_feature_groups,
    unavailable_expected_features,
)

observed = load_modeling_frame(view_name="strict_main", include_geometry=False, observed_only=True)
print(f"observed flood events: {len(observed):,}")
print(f"unique segments: {observed['segment_id'].nunique():,}")
display(summarize_feature_groups(observed))
display(unavailable_expected_features(observed))
display(feature_catalog(observed).head(40))

observed flood events: 34,870
unique segments: 17,053


,feature_group,expected_features,available_features,mean_non_null_rate,missing_features
0,event_time,8,8,0.998824,0
1,extra,18,18,0.999471,0
2,governance,3,3,1.000000,0
3,hydrometeorology,6,6,0.987104,0
4,infrastructure,4,4,0.500000,0
5,network,9,9,0.444444,0
6,socioeconomic,4,4,0.983589,0
7,spatial_controls,5,5,1.000000,0
8,targets,4,4,0.997648,0
9,terrain_coastal,8,8,0.999986,0


,feature_group,feature,available,dtype,non_null_rate,n_unique


,feature_group,feature,available,dtype,non_null_rate,n_unique
0,event_time,day_of_week,True,Int64,1.000000,7
1,event_time,duration,True,float64,1.000000,7452
2,event_time,end,True,datetime64[us],0.990594,31625
3,event_time,hour,True,Int64,1.000000,24
4,event_time,month,True,Int64,1.000000,12
5,event_time,season,True,string,1.000000,4
6,event_time,start,True,datetime64[us],1.000000,32123
7,event_time,storm_event_id,True,string,1.000000,117
8,extra,catch_basin_nearest_ft,True,float64,1.000000,16985
9,extra,component_size,True,int64,1.000000,52


In [ ]:
selection_df, archetypes, anomalies = evaluate_clustering_algorithms(observed)

display(
    selection_df.sort_values(
        ["feature_set", "silhouette_score", "calinski_harabasz_score"],
        ascending=[True, False, False],
        kind="stable",
    ).groupby("feature_set", as_index=False).head(8)
)
display(archetypes.head())
display(anomalies.head())

print(f"saved: {CLUSTERING_MODEL_SELECTION_PATH}")
print(f"saved: {CLUSTERING_ARCHETYPES_PATH}")
print(f"saved: {ANOMALY_SCORES_PATH}")

In [ ]:
combined_features = clustering_feature_sets(observed)["combined"]
pca_points = pca_embedding(observed, combined_features)
pca_plot = pca_points.merge(
    archetypes[["event_id", "combined_cluster_id", "cluster_label", "intensity", "resolution"]],
    on="event_id",
    how="left",
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
scatter = axes[0].scatter(
    pca_plot["pc1"],
    pca_plot["pc2"],
    c=pd.to_numeric(pca_plot["combined_cluster_id"], errors="coerce"),
    cmap="tab10",
    s=10,
    alpha=0.7,
)
axes[0].set_title("PCA Projection Colored by Combined Cluster")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
plt.colorbar(scatter, ax=axes[0], shrink=0.8)

axes[1].hist(pd.to_numeric(anomalies["anomaly_score"], errors="coerce").dropna(), bins=40, color="#DC2626", alpha=0.85)
axes[1].set_title("Anomaly Score Distribution")
axes[1].set_xlabel("ensemble anomaly score")
axes[1].set_ylabel("events")
plt.show()

In [ ]:
profile_cols = [
    column
    for column in [
        "max_tide",
        "prec_depth_total",
        "prec_duration_total",
        "elevation",
        "shore_dist",
        "edge_betweenness",
        "travel_time",
        "census_poverty_rate",
        "census_median_household_income",
        "intensity",
        "resolution",
    ]
    if column in archetypes.columns
]
cluster_profiles = archetypes.groupby("combined_cluster_id")[profile_cols].mean(numeric_only=True)
cluster_counts = archetypes["combined_cluster_id"].value_counts(dropna=False).sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
cluster_counts.plot.bar(ax=axes[0], color="#2563EB")
axes[0].set_title("Cluster Size Distribution")
axes[0].set_xlabel("combined_cluster_id")
axes[0].set_ylabel("events")

im = axes[1].imshow(cluster_profiles.to_numpy(dtype=float), aspect="auto", cmap="viridis")
axes[1].set_title("Cluster Profile Heatmap")
axes[1].set_xticks(range(len(cluster_profiles.columns)))
axes[1].set_xticklabels(cluster_profiles.columns, rotation=90)
axes[1].set_yticks(range(len(cluster_profiles.index)))
axes[1].set_yticklabels(cluster_profiles.index.astype(str))
plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.show()

display(cluster_profiles)
display(
    archetypes.groupby(["combined_cluster_id", "cluster_label"], as_index=False)
    .agg(
        n_events=("event_id", "size"),
        mean_intensity=("intensity", "mean"),
        mean_resolution_hours=("resolution", "mean"),
    )
)

NameError: name 'archetypes' is not defined

In [ ]:
try:
    geo = pd.read_parquet(CLUSTERING_ARCHETYPES_GEOPARQUET_PATH).to_crs(2263)
    top_anomalies = anomalies.head(250)[["event_id"]].merge(geo, on="event_id", how="left")

    fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
    geo.plot(ax=axes[0], column="combined_cluster_id", categorical=True, legend=True, linewidth=0.8, cmap="tab10")
    axes[0].set_title("Observed Events by Cluster")
    axes[0].set_axis_off()

    geo.plot(ax=axes[1], color="#94A3B8", linewidth=0.5, alpha=0.35)
    top_anomalies.plot(ax=axes[1], color="#DC2626", linewidth=1.2, alpha=0.85)
    axes[1].set_title("Top Anomalous Events")
    axes[1].set_axis_off()
    plt.show()
except Exception as exc:
    print(f"Spatial QA/QC plot skipped: {exc}")